# Phase 2 — Claim Extraction

This notebook covers the first piece of **Phase 2**: turning raw report text
into a list of checkable claims. The rest of Phase 2 (RAG chain / verdict
logic, `POST /verify`, Airflow DAG) isn't built yet — see
[docs/phase-2-rag-pipeline.md](../docs/phase-2-rag-pipeline.md) for the plan.

```mermaid
flowchart LR
    TEXT[Report text]
    LLM[Claim Extractor LLM]
    CLAIMS["Claims: entity, metric, value, date"]

    TEXT --> LLM --> CLAIMS

    CLAIMS -.->|next: not built yet| RAG[RAG chain + verdict]
```


In [1]:
import sys
sys.path.insert(0, "..")

## Claim extractor — structured output, not free-text parsing

`src/claim_extractor.py` asks the LLM for a typed `ClaimList` (via `ChatOpenAI(...).with_structured_output(ClaimList)`), not a text blob to regex apart. Each `Claim` has `text` (verbatim sentence), `entity`, `metric`, `value`, `date` — independently checkable against a source later.

The system prompt tells the model to skip opinions and forward-looking statements, keeping only claims with an entity + a number attached.

In [2]:
from src.claim_extractor import extract_claims

### Example: a snippet from a business report

Two checkable numeric claims (Apple revenue/net income, Walmart revenue), mixed with two opinion sentences that shouldn't survive extraction.

In [3]:
report_text = """Apple Inc. delivered another strong year. Revenue for fiscal year ending 2025-09-27 reached $416,161,000,000, while net income came in at $112,010,000,000.
Management remains optimistic about the upcoming product cycle. Separately, Walmart reported revenue of $713,163,000,000 for fiscal year ending 2026-01-31. 
Analysts believe the retail sector will keep growing."""

print(report_text)

Apple Inc. delivered another strong year. Revenue for fiscal year ending 2025-09-27 reached $416,161,000,000, while net income came in at $112,010,000,000.
Management remains optimistic about the upcoming product cycle. Separately, Walmart reported revenue of $713,163,000,000 for fiscal year ending 2026-01-31. 
Analysts believe the retail sector will keep growing.


In [4]:
claims = extract_claims(report_text)
for c in claims:
    print(c)

text='Revenue for fiscal year ending 2025-09-27 reached $416,161,000,000, while net income came in at $112,010,000,000.' entity='Apple Inc.' metric='Revenue' value=416161000000.0 date='fiscal year ending 2025-09-27'
text='Revenue for fiscal year ending 2025-09-27 reached $416,161,000,000, while net income came in at $112,010,000,000.' entity='Apple Inc.' metric='Net income' value=112010000000.0 date='fiscal year ending 2025-09-27'
text='Separately, Walmart reported revenue of $713,163,000,000 for fiscal year ending 2026-01-31.' entity='Walmart' metric='Revenue' value=713163000000.0 date='fiscal year ending 2026-01-31'


Both opinion sentences ("Management remains optimistic...", "Analysts believe...") were correctly dropped — no entity+number attached. All three extracted claims have a clean `entity`/`metric`/`value`/`date`, ready to look up against the Phase 1 knowledge base.

**Rough edge:** the first two claims share the same `text` — the model split one sentence containing two numbers ("revenue... while net income...") into two claims but kept the whole sentence as `text` for both, instead of the specific clause. Doesn't break the structured fields (each is independently correct), but worth a tighter prompt if per-claim quoting matters later.

## Model choice: OpenRouter, with a local Ollama fallback

Default is OpenRouter's free tier (`LLM_PROVIDER=openrouter`, `LLM_MODEL=tencent/hy3:free`) — $0 cost, fits a capstone with no LLM budget.

Originally tried `nvidia/nemotron-3-ultra-550b-a55b:free`: every call failed with `DEGRADED function cannot be invoked` — confirmed with a bare `openai` client too, so it's an outage on the provider's side, not a bug here. Swapped to `tencent/hy3:free`, which works cleanly with structured output (see extraction above).

`src/config.py` also supports `LLM_PROVIDER=ollama` — routes to a local Ollama model (`ornith:latest`) via its OpenAI-compatible endpoint
(`http://localhost:11434/v1`, dummy API key). No API key, no rate limits, works offline. Useful as a fallback when the OpenRouter free tier is degraded or rate-limited — not as the default, since it's CPU-only on this machine (~20s/call once warm, 2+ min cold start).

In [5]:
from langchain_openai import ChatOpenAI
from src.claim_extractor import ClaimList, SYSTEM_PROMPT

ollama_llm = ChatOpenAI(
    model="ornith:latest",
    api_key="ollama",
    base_url="http://localhost:11434/v1",
    temperature=0,
).with_structured_output(ClaimList)

result = ollama_llm.invoke([("system", SYSTEM_PROMPT), ("user", report_text)])
for c in result.claims:
    print(c)

text='Revenue for fiscal year ending 2025-09-27 reached $416,161,000,000' entity='Apple Inc.' metric='revenue' value=416161000000.0 date='fiscal year ending 2025-09-27'
text='net income came in at $112,010,000,000' entity='Apple Inc.' metric='net income' value=112010000000.0 date='fiscal year ending 2025-09-27'
text='Walmart reported revenue of $713,163,000,000 for fiscal year ending 2026-01-31' entity='Walmart' metric='revenue' value=713163000000.0 date='fiscal year ending 2026-01-31'


Same three claims, correctly split, and this time each `text` is the exact clause ("Revenue for..." vs "net income came in at...") instead of the shared full sentence OpenRouter's `tencent/hy3:free` produced above — a nice reminder that "free tier" and "good enough" don't always point at the same model, and it's worth A/B-ing more than one before locking in a default.

## What's next

Claim extraction is the first piece of Phase 2. Still open (see
[docs/phase-2-rag-pipeline.md](../docs/phase-2-rag-pipeline.md)):

- Verdict logic (LLM-as-judge) wired to `hybrid_search` from Phase 1/3
- `POST /verify` in `src/api.py`
- Airflow DAG for scheduled daily ingestion

[← Back to README](../README.md)